# 02 · Match-info files → a validated `matches` table

**Goal:** turn the 1,243 `*_info.csv` files into a one-row-per-match `matches` table, prove it correct against the ball-by-ball file, then use it to answer what notebook 01 had to leave open.

**Plan:** read every line into a lossless *long* table → survey the keys → write the survey into a schema → build `matches` → validate → close notebook 01's open items.

**Rules:** raw strings only — no renaming, standardising or inferred flags (cleaning is a later pass). The parser lives in `src/ipl_eda/info_files.py`; this notebook shows what it does and why.

**Contents:** Part B parse · Part C prove it · Part D close notebook 01's open items · what this notebook supersedes

## Part B — Parse

### B1 · Setup

In [1]:
%load_ext autoreload
%autoreload 2

import tempfile
from pathlib import Path

import numpy as np
import pandas as pd

from ipl_eda import info_files, tables   # works because `pip install -r requirements.txt` installed the project (editable)

pd.set_option("display.max_rows", None)
pd.set_option("display.max_columns", None)
pd.set_option("display.width", 200)
pd.set_option("display.max_colwidth", 80)

# Works whether Jupyter starts in notebooks/ (the default) or in the repo root.
RAW = next((p for p in [Path("../data/procured/cricsheet"), Path("data/procured/cricsheet")] if p.exists()), None)
if RAW is None:
    raise FileNotFoundError("data/procured/cricsheet not found - see README.md")
PROCESSED = RAW.parents[1] / "processed"
print("raw folder:", RAW, "| info files:", len(list(RAW.glob("*_info.csv"))))

raw folder: ../data/procured/cricsheet | info files: 1243


### B2 · What an info file looks like

Each file is a list of lines, not a table: a `version` line, then `info,<key>,<value>…` lines. A key can repeat (`team`, `umpire`), carry two or three values (`player`, `registry`), or be absent. One file, trimmed:

In [2]:
sample = RAW / "1527685_info.csv"
lines = sample.read_text(encoding="utf-8").splitlines()
print("\n".join(lines[:22]))
print("...")
print(next(l for l in lines if l.startswith("info,player")))
print(next(l for l in lines if l.startswith("info,registry")))

version,2.3.0
info,balls_per_over,6
info,team,Kolkata Knight Riders
info,team,Punjab Kings
info,team_type,club
info,gender,male
info,season,2026
info,date,2026/04/06
info,event,Indian Premier League
info,match_id,1527685
info,match_type,T20
info,overs,20
info,match_number,12
info,venue,"Eden Gardens, Kolkata"
info,city,Kolkata
info,toss_winner,Kolkata Knight Riders
info,toss_decision,bat
info,umpire,A Bengeri
info,umpire,J Madanagopal
info,reserve_umpire,UV Gandhe
info,tv_umpire,R Pandit
info,match_referee,J Srinath
...
info,player,Kolkata Knight Riders,AM Rahane
info,registry,people,A Bengeri,c109497d


In [3]:
# Why pd.read_csv is the wrong tool: the first line has 2 fields, the next ones 3 or more.
try:
    pd.read_csv(sample)
except Exception as err:
    print(type(err).__name__, "-", str(err).splitlines()[0])

ParserError - Error tokenizing data. C error: Expected 3 fields in line 24, saw 4


### B3 · Read everything into a lossless long table

One row per info line: `match_id`, `key`, and up to three value columns (left empty where a line has fewer). Nothing is interpreted. The `version` line is not an info line, so it is returned separately.

In [4]:
long, versions = info_files.read_info_folder(RAW)

n_lines = sum(len(p.read_text(encoding="utf-8").splitlines()) for p in RAW.glob("*_info.csv"))
print(f"files: {long['match_id'].nunique():,}   info lines (long-table rows): {len(long):,}   version lines: {len(versions):,}")
print("lossless check - lines in the files == long rows + version lines:", n_lines == len(long) + len(versions))
print("version values:", versions.value_counts().to_dict())
long.head(8)

files: 1,243   info lines (long-table rows): 94,447   version lines: 1,243
lossless check - lines in the files == long rows + version lines: True
version values: {'2.3.0': 1243}


,match_id,key,value1,value2,value3
0,1082591,balls_per_over,6,NaN,NaN
1,1082591,team,Sunrisers Hyderabad,NaN,NaN
2,1082591,team,Royal Challengers Bangalore,NaN,NaN
3,1082591,team_type,club,NaN,NaN
4,1082591,gender,male,NaN,NaN
5,1082591,season,2017,NaN,NaN
6,1082591,date,2017/04/05,NaN,NaN
7,1082591,event,Indian Premier League,NaN,NaN


### B4 · Survey the keys

For every key: in how many matches it appears, min/max rows per match, and the most value fields on any line. Then classify it — **constant** (same value in every match), **exactly one** per match, **optional** (at most one row, not in every match) or **repeating** (more than one row in some match).

In [5]:
rows = long.groupby(["match_id", "key"]).size().rename("rows").reset_index()
fields = long[["value1", "value2", "value3"]].notna().sum(axis=1)

survey = rows.groupby("key")["rows"].agg(n_matches="size", min_rows="min", max_rows="max")
survey["max_value_fields"] = fields.groupby(long["key"]).max()
survey["min_value_fields"] = fields.groupby(long["key"]).min()
survey["distinct_values"] = long.groupby("key")[["value1", "value2", "value3"]].apply(lambda g: len(g.drop_duplicates()))

def kind(row):
    if row["max_rows"] > 1:
        return "repeating"
    if row["n_matches"] < len(versions):
        return "optional"
    return "constant" if row["distinct_values"] == 1 else "exactly one"

survey["kind"] = survey.apply(kind, axis=1)
survey.sort_values(["kind", "n_matches"], ascending=[True, False])

,n_matches,min_rows,max_rows,max_value_fields,min_value_fields,distinct_values,kind
key,,,,,,,
balls_per_over,1243,1,1,1,1,1,constant
event,1243,1,1,1,1,1,constant
gender,1243,1,1,1,1,1,constant
match_type,1243,1,1,1,1,1,constant
overs,1243,1,1,1,1,1,constant
team_type,1243,1,1,1,1,1,constant
city,1243,1,1,1,1,38,exactly one
match_id,1243,1,1,1,1,1243,exactly one
match_referee,1243,1,1,1,1,32,exactly one


**Reading the survey**
- **31 keys.** Every line is `version` or `info`; the `info` lines have 3, 4 or 5 fields, so a key carries 1, 2 or 3 value fields (`player` has 2: team and name; `registry` has 3: `people`, name, id).
- **Constant (6):** `balls_per_over`, `event`, `gender`, `match_type`, `overs`, `team_type`. Same value in all 1,243 matches — they describe the archive, not a match, so they are listed (next cell) rather than turned into columns.
- **Exactly one per match (7):** `city`, `match_id`, `match_referee`, `season`, `toss_decision`, `toss_winner`, `venue`. Every match has a `city` line, but the line is **blank in 51 of them** (`info,city,`) — see the notes under B6.
- **Optional (12):** the result fields (`winner`, `winner_runs`, `winner_wickets`, `outcome`, `method`, `eliminator`), `match_number` (missing in 74 matches), `player_of_match`, `target_runs`, `target_overs` (missing in 6) and two officials.
- **Repeating (6):** `team` (always 2), `umpire` (always 2), `date` (1 or 2), `player`, `registry` and `super_over` (2 or 4 lines in 16 matches).
- **Two keys carry two value fields — `target_runs` and `target_overs`.** The first field is the innings number and it is always `2`: a target is only recorded for the chase. The `matches` column keeps the second field (the value); the first is asserted below.

In [6]:
# The constant keys: listed and asserted, not silently ignored.
constant = survey.index[survey["kind"] == "constant"]
assert set(constant) == set(info_files.CONSTANT_KEYS)
for key in constant:
    assert len(long.loc[long["key"] == key, ["value1", "value2", "value3"]].drop_duplicates()) == 1

# The first field of the two-field keys is always the innings number 2.
two_fields = long[long["key"].isin(["target_runs", "target_overs"])]
assert (two_fields["value1"] == "2").all()

long[long["key"].isin(constant)].drop_duplicates("key")[["key", "value1"]].set_index("key")

,value1
key,
balls_per_over,6
team_type,club
gender,male
event,Indian Premier League
match_type,T20
overs,20


### B5 · Turn the survey into a schema

The parser keeps the survey as `SCHEMA`: for each accepted key, the fewest and the most rows it may have in one match, and how many value fields a line must have. A key with a minimum of 1 or more is *required*: a match without it stops the build. Anything outside the schema must raise — never be dropped, overwritten or guessed.

In [7]:
schema = pd.DataFrame(info_files.SCHEMA, index=["min_rows_per_match", "max_rows_per_match", "value_fields"]).T.astype("Int64")
schema      # min 0 = optional key; max <NA> = a list that may repeat freely

,min_rows_per_match,max_rows_per_match,value_fields
balls_per_over,1,1,1
event,1,1,1
gender,1,1,1
match_type,1,1,1
overs,1,1,1
team_type,1,1,1
match_id,1,1,1
season,1,1,1
venue,1,1,1
city,1,1,1


In [8]:
# The schema is typed by hand, so check it against the survey instead of trusting it.
declared = schema.reindex(survey.index)
expected_min = survey["min_rows"].where(survey["n_matches"] == len(versions), 0)      # a key absent from some match has minimum 0
bounded = declared["max_rows_per_match"].notna()
assert set(schema.index) == set(survey.index), "schema and survey cover different keys"
assert (declared["min_rows_per_match"] == expected_min).all(), "a minimum differs from the survey"
assert (declared.loc[bounded, "max_rows_per_match"] == survey.loc[bounded, "max_rows"]).all(), "a maximum differs from the survey"
assert (survey["min_value_fields"] == survey["max_value_fields"]).all() and (declared["value_fields"] == survey["max_value_fields"]).all()
print(f"all {len(schema)} schema entries agree with the survey ({int((~bounded).sum())} lists have no maximum)")

all 31 schema entries agree with the survey (2 lists have no maximum)


In [9]:
info_files.check_schema(long)
print("the real data passes the schema")

# Deliberately malformed inputs. Each one must be refused.
some_match = long["match_id"].iloc[0]
bad_inputs = {
    "unknown key": pd.concat([long, pd.DataFrame([(some_match, "weather", "rain", None, None)], columns=long.columns)]),
    "repeated single-valued key": pd.concat([long, long[(long["match_id"] == some_match) & (long["key"] == "season")]]),
    "wrong number of value fields": pd.concat([long, pd.DataFrame([(some_match, "toss_decision", "bat", "field", None)], columns=long.columns)]).drop(
        long[(long["match_id"] == some_match) & (long["key"] == "toss_decision")].index),
    "required key missing from one match": long.drop(long[(long["match_id"] == some_match) & (long["key"] == "toss_winner")].index),
    "second team line missing": long.drop(long[(long["match_id"] == some_match) & (long["key"] == "team")].index[-1]),
}
for name, frame in bad_inputs.items():
    try:
        info_files.check_schema(frame)
    except ValueError as err:
        print(f"{name:<36} -> ValueError: {err}")

# An empty folder must raise too, instead of returning empty tables.
with tempfile.TemporaryDirectory() as empty_folder:
    try:
        info_files.read_info_folder(empty_folder)
    except FileNotFoundError as err:
        print(f"{'empty folder':<36} -> FileNotFoundError: {str(err).replace(empty_folder, '<empty folder>')}")

the real data passes the schema
unknown key                          -> ValueError: unknown info key(s): ['weather']
repeated single-valued key           -> ValueError: key 'season' has more than 1 row(s) in 1 match(es), first: match 1082591 has 2
wrong number of value fields         -> ValueError: 1 line(s) with the wrong number of value fields, first: match 1082591, key 'toss_decision'
required key missing from one match  -> ValueError: key 'toss_winner' has fewer than 1 row(s) in 1 match(es), first: match 1082591 has 0
second team line missing             -> ValueError: key 'team' has fewer than 2 row(s) in 1 match(es), first: match 1082591 has 1
empty folder                         -> FileNotFoundError: no *_info.csv files in <empty folder> - wrong folder?


### B6 · Build `matches`

One column per informative single-valued key, named after the key. `team` becomes `team1` / `team2` (in file order) and `date` becomes `start_date` / `end_date`. Text stays raw; dates are parsed with an explicit format. Every key is either a column or explicitly left out.

In [10]:
matches = info_files.build_matches(long)

mapped = set(info_files.SINGLE_COLUMNS) | set(info_files.TWO_FIELD_COLUMNS) | {"team", "date"}
left_out = set(info_files.CONSTANT_KEYS) | set(info_files.LEFT_OUT_KEYS)
assert mapped | left_out == set(long["key"]) and not mapped & left_out      # every key is accounted for exactly once

print("mapped to columns:", sorted(mapped))
print("constant (listed above):", sorted(info_files.CONSTANT_KEYS))
print("left out on purpose:", sorted(info_files.LEFT_OUT_KEYS))
print(f"\nmatches: {matches.shape[0]:,} rows x {matches.shape[1]} columns | match_id unique: {matches['match_id'].is_unique}")
matches.dtypes.rename("dtype").to_frame()

mapped to columns: ['city', 'date', 'eliminator', 'match_number', 'method', 'outcome', 'player_of_match', 'season', 'target_overs', 'target_runs', 'team', 'toss_decision', 'toss_winner', 'venue', 'winner', 'winner_runs', 'winner_wickets']
constant (listed above): ['balls_per_over', 'event', 'gender', 'match_type', 'overs', 'team_type']
left out on purpose: ['match_id', 'match_referee', 'player', 'registry', 'reserve_umpire', 'super_over', 'tv_umpire', 'umpire']

matches: 1,243 rows x 20 columns | match_id unique: True


,dtype
match_id,int64
season,str
start_date,datetime64[us]
end_date,datetime64[us]
team1,str
team2,str
venue,str
city,str
match_number,Int64
toss_winner,str


In [11]:
matches.head(5)

,match_id,season,start_date,end_date,team1,team2,venue,city,match_number,toss_winner,toss_decision,player_of_match,winner,winner_runs,winner_wickets,outcome,method,eliminator,target_runs,target_overs
0,335982,2007/08,2008-04-18,NaT,Royal Challengers Bangalore,Kolkata Knight Riders,M Chinnaswamy Stadium,Bangalore,1,Royal Challengers Bangalore,field,BB McCullum,Kolkata Knight Riders,140,<NA>,NaN,NaN,NaN,223,20
1,335983,2007/08,2008-04-19,NaT,Kings XI Punjab,Chennai Super Kings,"Punjab Cricket Association Stadium, Mohali",Chandigarh,2,Chennai Super Kings,bat,MEK Hussey,Chennai Super Kings,33,<NA>,NaN,NaN,NaN,241,20
2,335984,2007/08,2008-04-19,NaT,Delhi Daredevils,Rajasthan Royals,Feroz Shah Kotla,Delhi,3,Rajasthan Royals,bat,MF Maharoof,Delhi Daredevils,<NA>,9,NaN,NaN,NaN,130,20
3,335985,2007/08,2008-04-20,NaT,Mumbai Indians,Royal Challengers Bangalore,Wankhede Stadium,Mumbai,5,Mumbai Indians,bat,MV Boucher,Royal Challengers Bangalore,<NA>,5,NaN,NaN,NaN,166,20
4,335986,2007/08,2008-04-20,NaT,Kolkata Knight Riders,Deccan Chargers,Eden Gardens,Kolkata,4,Deccan Chargers,bat,DJ Hussey,Kolkata Knight Riders,<NA>,5,NaN,NaN,NaN,111,20


In [12]:
matches.isna().sum().rename("missing values").to_frame()

,missing values
match_id,0
season,0
start_date,0
end_date,1241
team1,0
team2,0
venue,0
city,51
match_number,74
toss_winner,0


In [13]:
PROCESSED.mkdir(exist_ok=True)
matches.to_csv(PROCESSED / "matches.csv", index=False)

# Write-then-read: the typed loader must give back exactly the table that was written - values AND dtypes.
loaded = tables.read_matches(PROCESSED / "matches.csv")
pd.testing.assert_frame_equal(loaded, matches)
print(f"saved {PROCESSED / 'matches.csv'}: {len(loaded):,} rows; read_matches() returns the in-memory table exactly, dtypes included")

# What a default pd.read_csv would have done instead (only the columns whose type changes):
default = pd.read_csv(PROCESSED / "matches.csv")
changed = pd.DataFrame({"in memory = read_matches": matches.dtypes.astype(str), "default pd.read_csv": default.dtypes.astype(str)})
print("default read: target_overs values", default["target_overs"].dropna().unique()[:6].tolist(), "- the text '20' and '9.2' became floats")
changed[changed.iloc[:, 0] != changed.iloc[:, 1]]

saved ../data/processed/matches.csv: 1,243 rows; read_matches() returns the in-memory table exactly, dtypes included
default read: target_overs values [20.0, 16.0, 18.0, 8.0, 6.0, 9.2] - the text '20' and '9.2' became floats


,in memory = read_matches,default pd.read_csv
start_date,datetime64[us],str
end_date,datetime64[us],str
match_number,Int64,float64
winner_runs,Int64,float64
winner_wickets,Int64,float64
target_runs,Int64,float64
target_overs,str,float64


**Notes on the build**
- **Grain.** A one-row-per-match table can only hold what occurs once per match, or a small fixed number of times: `team` is always exactly two, so it becomes `team1` / `team2`. `player` (22–25 lines per match) and `registry` (26–32) cannot be columns, and the officials and super-over innings are left out too. All of them stay in `long`; if a question needs them they get their own tables.
- **Nullable integers.** `match_number` (74 missing), `winner_runs`, `winner_wickets` and `target_runs` are `Int64`: plain `int64` can't hold a missing value, and `float64` would turn 56 into `56.0`.
- **Raw text.** `season`, `venue`, `city`, team names and `target_overs` are exactly as written. `target_overs` stays a string because Cricsheet writes overs as `over.ball` (section C3 checks this).
- **Missing values** are `NaN` / `NaT` / `<NA>` — not `None` — so test with `pd.isna`, never `is None`. A blank field counts as missing: `city` is blank (`info,city,`) in 51 matches, all at the Dubai and Sharjah grounds (2014 and 2020/21); `long` keeps the raw empty string, `matches` has a missing value.
- **Types in a CSV.** A CSV carries no types, so `tables.read_matches()` states them: text stays text (`target_overs` too), margins are `Int64`, dates are dates. The first version of this check found a real difference: the blank `city` fields are `""` in memory but missing after a round trip, which is why blanks become missing in `build_matches`.
- `end_date` is filled for only 2 matches (play ran past midnight); for every other match there is one date line.

## Part C — Prove it

Parsed is not validated. Each check compares the table with an independent source (the deliveries file) or with a rule the key survey suggests, and every result lands in one table at the end of this part.

### C1 · What the deliveries file says about each match

In [14]:
cols = ["match_id", "season", "start_date", "venue", "innings", "batting_team", "bowling_team", "runs_off_bat", "extras", "wides", "noballs"]
deliveries = pd.read_csv(RAW / "all_matches.csv", usecols=cols, low_memory=False)

# One row per match with the facts both files should agree on. Dates are compared as dates, not strings
# (2017/04/05 in the info files, 2017-04-05 in the deliveries).
from_deliveries = (
    deliveries.drop_duplicates("match_id")[["match_id", "season", "start_date", "venue"]]
    .assign(start_date=lambda d: pd.to_datetime(d["start_date"], format="%Y-%m-%d"))
)
first_innings_teams = deliveries[deliveries["innings"] == 1].drop_duplicates("match_id").set_index("match_id")[["batting_team", "bowling_team"]]

# Runs per innings (runs off the bat + extras) and who batted.
runs = (
    deliveries.assign(runs=deliveries["runs_off_bat"] + deliveries["extras"])
    .groupby(["match_id", "innings"]).agg(runs=("runs", "sum"), team=("batting_team", "first")).reset_index()
)
inn1 = runs[runs["innings"] == 1].set_index("match_id")[["runs", "team"]].add_prefix("inn1_")
inn2 = runs[runs["innings"] == 2].set_index("match_id")[["runs", "team"]].add_prefix("inn2_")

both = (
    matches.merge(from_deliveries, on="match_id", suffixes=("", "_deliveries"))
    .join(inn1, on="match_id").join(inn2, on="match_id")
)
print(f"matches: {len(matches):,} | deliveries file: {deliveries['match_id'].nunique():,} | in both: {len(both):,}")
both[["match_id", "season", "season_deliveries", "start_date", "start_date_deliveries", "inn1_runs", "inn2_runs"]].head(3)

matches: 1,243 | deliveries file: 1,243 | in both: 1,243


,match_id,season,season_deliveries,start_date,start_date_deliveries,inn1_runs,inn2_runs
0,335982,2007/08,2007/08,2008-04-18,2008-04-18,222,82.0
1,335983,2007/08,2007/08,2008-04-19,2008-04-19,240,207.0
2,335984,2007/08,2007/08,2008-04-19,2008-04-19,129,132.0


### C2 · Two things to settle first

**Who won?** `winner` is blank in 25 matches, and a blank is not one thing. How `winner`, `outcome`, `eliminator` and `method` combine:

In [15]:
pattern = (
    matches.assign(
        winner_recorded=matches["winner"].notna(),
        outcome=matches["outcome"].fillna("-"),
        eliminator_recorded=matches["eliminator"].notna(),
        method=matches["method"].fillna("-"),
    )
    .groupby(["winner_recorded", "outcome", "eliminator_recorded", "method"]).size().rename("matches").to_frame()
)

# Working definition used in this notebook only (matches.csv stays raw): who won = winner, else the super-over winner.
who_won = matches["winner"].fillna(matches["eliminator"])
print(f"somebody won: {who_won.notna().sum():,} | nobody (no result): {who_won.isna().sum()} | rows with a plain `winner`: {matches['winner'].notna().sum():,}")
pattern

somebody won: 1,234 | nobody (no result): 9 | rows with a plain `winner`: 1,218


matches
winner_recorded outcome   eliminator_recorded method         
False           no result False               -             9
                tie       True                -            16
True            -         False               -          1195
                                              D/L          23

- **1,195** matches: a `winner` and nothing else — a plain result.
- **23**: a `winner` plus `method = D/L` — a rain-affected result.
- **16**: no `winner`, `outcome = tie` and an `eliminator` — the scores were level and a **super over** decided it. The `eliminator` field names that winner.
- **9**: no `winner`, `outcome = no result` — nobody won.

So "no winner" is two different situations (16 super-over decisions, 9 no results). `winner.notna()` would silently drop the 16 super-over decisions from a win count, so from here on **who won = `winner`, else `eliminator`, else nobody**.

**Does the order of `team1` / `team2` mean anything?** Test it rather than assume it:

In [16]:
tests = pd.DataFrame({
    "team1 batted first": (both["team1"] == both["inn1_team"]).astype(float),
    "team1 won the toss": (both["team1"] == both["toss_winner"]).astype(float),
    "team1 is first alphabetically": (both["team1"] < both["team2"]).astype(float),
    "team1 won the match": (both["team1"] == both["winner"]).astype(float).where(both["winner"].notna()),
})
by_year = (both["team1"] == both["inn1_team"]).groupby(both["start_date"].dt.year).mean().round(2)
print("share of matches where team1 batted first, by year:", by_year.to_dict())
era = np.where(both["start_date"].dt.year < 2018, "2008-2017", "2018-2026")
by_era = tests.groupby(era).mean().round(3).T
by_era["all matches"] = tests.mean().round(3)
print("matches per era:", pd.Series(era).value_counts().sort_index().to_dict(), "| share of matches where each statement is true:")
by_era

share of matches where team1 batted first, by year: {2008: 0.55, 2009: 0.47, 2010: 0.52, 2011: 0.6, 2012: 0.5, 2013: 0.43, 2014: 0.55, 2015: 0.63, 2016: 0.58, 2017: 0.46, 2018: 1.0, 2019: 1.0, 2020: 1.0, 2021: 1.0, 2022: 1.0, 2023: 1.0, 2024: 1.0, 2025: 1.0, 2026: 1.0}
matches per era: {'2008-2017': 636, '2018-2026': 607} | share of matches where each statement is true:


,2008-2017,2018-2026,all matches
team1 batted first,0.528,1.000,0.759
team1 won the toss,0.483,0.239,0.364
team1 is first alphabetically,0.524,0.491,0.508
team1 won the match,0.538,0.458,0.499


**Reading the order.** From 2018 on `team1` is the team that **batted first in every match** (607 of 607); the by-year shares above show where the change happens (0.46 in 2017, 1.00 in every year from 2018), so the split is read from the data, not assumed. In 2008–2017 it is about half (53 %) — no pattern. In neither era is it reliably the toss winner, the alphabetically first team or the winner. So the order of the two team rows is not one convention across the archive: treat `team1` / `team2` as an **unordered pair**, and take "batted first" from the deliveries (innings 1), never from the team order.

### C3 · How are target overs written?

`target_overs` looks like a number (`20`, `16`, `9.2`). Match 392186 has `9.2`; compare it with what the deliveries file says about innings 2 of that match:

In [17]:
match_92 = int(matches.loc[matches["target_overs"] == "9.2", "match_id"].iloc[0])
innings2 = deliveries[(deliveries["match_id"] == match_92) & (deliveries["innings"] == 2)]
legal_balls = int((innings2["wides"].isna() & innings2["noballs"].isna()).sum())
print(f"match {match_92}: target_overs = '9.2'; legal balls bowled in innings 2 = {legal_balls} = {legal_balls // 6} overs and {legal_balls % 6} balls")
print("distinct target_overs:", sorted(matches["target_overs"].dropna().unique(), key=float))

match 392186: target_overs = '9.2'; legal balls bowled in innings 2 = 56 = 9 overs and 2 balls
distinct target_overs: ['5', '6', '8', '9', '9.2', '10', '11', '12', '13', '14', '15', '16', '17', '18', '19', '20']


`9.2` is **9 overs and 2 balls** (56 balls), written `over.ball` — not 9.2 as a decimal. That is why `target_overs` stays a raw string in `matches`; convert it only by splitting at the dot.

### C4 · The scoreboard check (across the two files)

For a match with a recorded `winner` and no `method`, the winner must be the team with the higher innings-1/innings-2 total (runs off the bat + extras), and for a win by runs `winner_runs` must equal the difference. Then the same comparison on the D/L matches:

In [18]:
decided = both[both["winner"].notna()].copy()
decided["higher_total_team"] = np.where(
    decided["inn1_runs"] > decided["inn2_runs"], decided["inn1_team"],
    np.where(decided["inn2_runs"] > decided["inn1_runs"], decided["inn2_team"], "level"),
)
decided["run_difference"] = (decided["inn1_runs"] - decided["inn2_runs"]).abs()

summary = []
for label, group in decided.groupby(decided["method"].fillna("no method")):
    by_runs = group[group["winner_runs"].notna()]
    summary.append({
        "results": label,
        "matches": len(group),
        "winner is not the higher total": int((group["winner"] != group["higher_total_team"]).sum()),
        "wins by runs": len(by_runs),
        "winner_runs != run difference": int((by_runs["winner_runs"] != by_runs["run_difference"]).sum()),
    })
pd.DataFrame(summary).set_index("results")

,matches,winner is not the higher total,wins by runs,winner_runs != run difference
results,,,,
D/L,23,14,13,13
no method,1195,0,545,0


In [19]:
# The 23 D/L matches: a rain-revised target explains what the raw totals cannot.
plain = decided[decided["method"].isna()]
dl = decided[decided["method"].notna()].copy()
dl["chaser_won"] = dl["winner"] == dl["inn2_team"]
dl["chaser_total_reached_target"] = dl["inn2_runs"] >= dl["target_runs"]

print("plain results where target_runs == innings-1 total + 1:", int((plain["target_runs"] == plain["inn1_runs"] + 1).sum()), "of", len(plain))
print("D/L results where target_runs == innings-1 total + 1  :", int((dl["target_runs"] == dl["inn1_runs"] + 1).sum()), "of", len(dl))
print("D/L results where 'chaser reached target_runs' predicts the winner:", int((dl["chaser_won"] == dl["chaser_total_reached_target"]).sum()), "of", len(dl))
dl[["match_id", "season", "winner", "inn1_runs", "inn2_runs", "target_runs", "target_overs", "winner_runs", "winner_wickets"]]

plain results where target_runs == innings-1 total + 1: 1195 of 1195
D/L results where target_runs == innings-1 total + 1  : 0 of 23
D/L results where 'chaser reached target_runs' predicts the winner: 23 of 23


,match_id,season,winner,inn1_runs,inn2_runs,target_runs,target_overs,winner_runs,winner_wickets
40,336022,2007/08,Kings XI Punjab,118,94.0,89,8,6,<NA>
43,336025,2007/08,Chennai Super Kings,149,55.0,53,8,3,<NA>
60,392183,2009,Delhi Daredevils,104,58.0,54,6,<NA>,10
63,392186,2009,Kolkata Knight Riders,158,79.0,69,9.2,11,<NA>
89,392214,2009,Chennai Super Kings,185,174.0,187,18,12,<NA>
192,501215,2011,Kochi Tuskers Kerala,131,135.0,135,17,<NA>,7
221,501245,2011,Kolkata Knight Riders,114,61.0,52,10,10,<NA>
231,501255,2011,Royal Challengers Bangalore,89,105.0,102,13,<NA>,4
429,733993,2014,Sunrisers Hyderabad,143,44.0,43,5,<NA>,8
477,829743,2015,Sunrisers Hyderabad,176,101.0,118,12,16,<NA>


**Reading the scoreboard check**
- All **1,195 plain results reconcile exactly**: the winner always has the higher total, every one of the 545 wins by runs equals the difference, and `target_runs` is the first-innings total + 1 every time — so `target_runs` has a clear meaning.
- The **23 D/L matches do not reconcile with raw totals**: in 14 the winner has the *lower* raw total, and none of the 13 wins by runs equals the raw difference. This is expected, not a parser bug: D/L replaces the target (`target_runs` is never first-innings total + 1 here — it is the revised target, set for the reduced `target_overs`). Measured against that revised target every result reconciles: the chasing team won exactly when its total reached `target_runs` (23 of 23).
- So raw-scoreboard checks are valid for plain results only; for D/L matches use `target_runs` and `target_overs`.

### C5 · The validation table

All checks in one table. `violations` counts the matches (or lines) that break a rule; `explained` is how many of those the checks above account for. The cell **fails** if any violation is unexplained. (The two `explained` figures for the D/L rules are typed here; notebook 03 turns them into computed rules.)

In [20]:
rules = []

def rule(check, name, broken, explained=0, why=""):
    """Record one rule. `broken` is a count or a boolean mask over matches."""
    count = int(broken) if np.isscalar(broken) else int(pd.Series(broken).fillna(False).astype(bool).sum())
    rules.append({"check": check, "rule": name, "violations": count, "explained": explained, "why": why})

# --- 9 identity
ids_info, ids_deliveries = set(matches["match_id"]), set(deliveries["match_id"])
file_ids = long[long["key"] == "match_id"]
rule("identity", "rows in matches != number of info files", len(matches) != len(versions))
rule("identity", "match_id appears twice in matches", matches["match_id"].duplicated().sum())
rule("identity", "id in the info files but not in all_matches.csv", len(ids_info - ids_deliveries))
rule("identity", "id in all_matches.csv but not in the info files", len(ids_deliveries - ids_info))
rule("identity", "the file's own match_id line differs from its file name", (file_ids["value1"].astype(int) != file_ids["match_id"]).sum())

# --- 10 shared facts: the two files must agree, match by match
t = first_innings_teams.reindex(both["match_id"]).reset_index(drop=True)
same_pair = (
    ((both["team1"] == t["batting_team"]) & (both["team2"] == t["bowling_team"]))
    | ((both["team1"] == t["bowling_team"]) & (both["team2"] == t["batting_team"]))
)
rule("shared facts", "season differs", both["season"] != both["season_deliveries"])
rule("shared facts", "venue differs", both["venue"] != both["venue_deliveries"])
rule("shared facts", "start date differs", both["start_date"] != both["start_date_deliveries"])
rule("shared facts", "the two teams differ (as a pair, either order)", ~same_pair)

# --- 11 internal rules
has_winner, has_outcome, has_elim = both["winner"].notna(), both["outcome"].notna(), both["eliminator"].notna()
for col in ["toss_winner", "winner", "eliminator"]:
    rule("internal", f"{col} is not one of the two teams", both[col].notna() & (both[col] != both["team1"]) & (both[col] != both["team2"]))
rule("internal", "winner and outcome are not exactly one-of", has_winner == has_outcome)
rule("internal", "eliminator is present exactly when outcome == tie", has_elim != (both["outcome"] == "tie"))
rule("internal", "a winner without exactly one of winner_runs / winner_wickets", has_winner & (both["winner_runs"].notna() == both["winner_wickets"].notna()))
rule("internal", "winner_runs / winner_wickets without a winner", ~has_winner & (both["winner_runs"].notna() | both["winner_wickets"].notna()))
# further rules suggested by the key survey
rule("internal", "toss_decision is not bat / field", ~both["toss_decision"].isin(["bat", "field"]))
rule("internal", "outcome is not tie / no result", has_outcome & ~both["outcome"].isin(["tie", "no result"]))
rule("internal", "end_date is before start_date", both["end_date"] < both["start_date"])
rule("internal", "(season, match_number) repeats", both.dropna(subset=["match_number"]).duplicated(["season", "match_number"]))
rule("internal", "a method without target_runs / target_overs", both["method"].notna() & (both["target_runs"].isna() | both["target_overs"].isna()))

# --- survey rules that need the deliveries file
super_info = set(long.loc[long["key"] == "super_over", "match_id"])
super_deliveries = set(deliveries.loc[deliveries["innings"] > 2, "match_id"])
rule("across files", "target recorded exactly when a 2nd innings exists", both["target_runs"].notna() != both["inn2_runs"].notna())
rule("across files", "super_over lines exactly when innings > 2 exist", len(super_info ^ super_deliveries))
rule("across files", "an eliminator without super_over lines", has_elim & ~both["match_id"].isin(super_info))

# --- 12 scoreboard
ties = both[both["outcome"] == "tie"]
rule("scoreboard", "plain results: winner is not the higher total", plain["winner"] != plain["higher_total_team"])
rule("scoreboard", "plain wins by runs: winner_runs != run difference", plain["winner_runs"].notna() & (plain["winner_runs"] != plain["run_difference"]))
rule("scoreboard", "plain results: target_runs != innings-1 total + 1", plain["target_runs"] != plain["inn1_runs"] + 1)
rule("scoreboard", "ties: innings 1 and 2 totals differ", ties["inn1_runs"] != ties["inn2_runs"])
rule("scoreboard", "D/L: winner is not the higher raw total", dl["winner"] != dl["higher_total_team"], explained=14,
     why="D/L revises the target, so raw totals are the wrong test")
rule("scoreboard", "D/L wins by runs: winner_runs != raw difference", dl["winner_runs"].notna() & (dl["winner_runs"] != dl["run_difference"]), explained=13,
     why="the D/L margin is measured against the revised target")
rule("scoreboard", "D/L: 'chaser reached target_runs' disagrees with the winner", dl["chaser_won"] != dl["chaser_total_reached_target"])

validation = pd.DataFrame(rules)
unexplained = validation[validation["violations"] != validation["explained"]]
assert unexplained.empty, f"unexplained violations:\n{unexplained}"
validation

,check,rule,violations,explained,why
0,identity,rows in matches != number of info files,0,0,
1,identity,match_id appears twice in matches,0,0,
2,identity,id in the info files but not in all_matches.csv,0,0,
3,identity,id in all_matches.csv but not in the info files,0,0,
4,identity,the file's own match_id line differs from its file name,0,0,
5,shared facts,season differs,0,0,
6,shared facts,venue differs,0,0,
7,shared facts,start date differs,0,0,
8,shared facts,"the two teams differ (as a pair, either order)",0,0,
9,internal,toss_winner is not one of the two teams,0,0,


## Part D — Close notebook 01's open items

### D1 · The short and single-innings matches (notebook 01, section 5)

In [21]:
per_match = deliveries.groupby("match_id").size().rename("deliveries")
innings_recorded = deliveries.groupby("match_id")["innings"].nunique().rename("innings_recorded")
interest = sorted(set(innings_recorded[innings_recorded == 1].index) | set(per_match.nsmallest(4).index) | {1178424})

(matches.set_index("match_id").join(per_match).join(innings_recorded)
 .loc[interest, ["season", "start_date", "team1", "team2", "deliveries", "innings_recorded", "outcome", "winner", "method", "target_overs", "target_runs"]]
 .sort_values("deliveries"))

,season,start_date,team1,team2,deliveries,innings_recorded,outcome,winner,method,target_overs,target_runs
match_id,,,,,,,,,,,
1527685,2026,2026-04-06,Kolkata Knight Riders,Punjab Kings,22,1,no result,NaN,NaN,NaN,<NA>
1178424,2019,2019-04-30,Royal Challengers Bangalore,Rajasthan Royals,51,2,no result,NaN,NaN,5,63
1473495,2025,2025-05-08,Punjab Kings,Delhi Capitals,63,1,no result,NaN,NaN,NaN,<NA>
501265,2011,2011-05-21,Delhi Daredevils,Pune Warriors,63,1,no result,NaN,NaN,NaN,<NA>
1359519,2023,2023-05-03,Lucknow Super Giants,Chennai Super Kings,118,1,no result,NaN,NaN,NaN,<NA>
1473492,2025,2025-05-05,Delhi Capitals,Sunrisers Hyderabad,123,1,no result,NaN,NaN,NaN,<NA>
829763,2015,2015-04-29,Royal Challengers Bangalore,Rajasthan Royals,128,1,no result,NaN,NaN,NaN,<NA>


In [22]:
# Result type against the length of the chase, for all 1,243 matches.
result = np.select(
    [matches["method"].notna(), matches["winner"].notna(), matches["eliminator"].notna()],
    ["won, D/L", "won, no method", "tie, decided by super over"], default="no result")
chase = np.select(
    [matches["target_overs"].isna(), matches["target_overs"] == "20"],
    ["no chase recorded", "20 overs"], default="fewer than 20 overs")
pd.crosstab(pd.Series(result, name="result"), pd.Series(chase, name="chase"))

chase,20 overs,fewer than 20 overs,no chase recorded
result,,,
no result,2,1,6
"tie, decided by super over",16,0,0
"won, D/L",0,23,0
"won, no method",1185,10,0


**Reading D1**
- **The short matches do share a result type.** All six single-innings matches and all four of the lowest-delivery matches (22, 51, 63, 63 deliveries) are recorded as `no result` — nobody won and no method was applied.
- **1178424** (51 deliveries): notebook 01 called it a reduced-overs match. The info file agrees — `target_overs = 5`, `target_runs = 63` (62 + 1) — and adds what notebook 01 could not see: the chase, stopped at 3.2 overs, **never produced a result**. The reading stands, with "and no result" added.
- A `no result` is not always a one-innings match: three of the nine (829813, 1178424, 1473481) have two innings.
- Most reduced-overs matches (`target_overs` below 20 — 34 in all) did finish: 23 by D/L, 10 with a plain result and no method at all (e.g. 598068, 8 overs each), one no result. So a short match with a clean result can carry no `method`; `method` only marks the shortened matches where D/L was applied.

### D2 · `match_number` gaps (notebook 01, section 3.1)

Notebook 01 flagged short seasons by eye and proposed testing them against the info files. That test can't work directly: the info files are one-to-one with the deliveries files (same 1,243 ids), so a match missing from one is missing from both. What the info files do hold is `match_number`, so the test is: does each season's numbering run 1..max without holes?

In [23]:
rows = []
for season, g in matches.groupby("season"):
    numbers = g["match_number"].dropna().astype(int)
    missing = sorted(set(range(1, numbers.max() + 1)) - set(numbers))
    rows.append({"season": season, "year": g["start_date"].min().year, "matches": len(g), "numbered": len(numbers),
                 "unnumbered": int(g["match_number"].isna().sum()), "max_number": numbers.max(),
                 "missing_numbers": missing, "n_missing": len(missing)})
season_gaps = pd.DataFrame(rows).sort_values("year").set_index("season")

in_notebook_01 = ["2007/08", "2009", "2011", "2015", "2017", "2024"]      # the six seasons flagged by eye in notebook 01, section 3.1
season_gaps["flagged in notebook 01"] = season_gaps.index.isin(in_notebook_01)
print("seasons with gaps:", season_gaps.index[season_gaps["n_missing"] > 0].tolist(), "| missing numbers in total:", int(season_gaps["n_missing"].sum()))
season_gaps

seasons with gaps: ['2007/08', '2009', '2011', '2012', '2015', '2017', '2024'] | missing numbers in total: 10


,year,matches,numbered,unnumbered,max_number,missing_numbers,n_missing,flagged in notebook 01
season,,,,,,,,
2007/08,2008,58,55,3,56,[47],1,True
2009,2009,57,54,3,56,"[7, 13]",2,True
2009/10,2010,60,56,4,56,[],0,False
2011,2011,73,69,4,70,[20],1,True
2012,2012,74,70,4,72,"[32, 34]",2,False
2013,2013,76,72,4,72,[],0,False
2014,2014,60,56,4,56,[],0,False
2015,2015,59,55,4,56,[25],1,True
2016,2016,60,56,4,56,[],0,False


In [24]:
# Where does each missing number sit in time? The matches either side of it.
dated = matches.dropna(subset=["match_number"]).set_index(["season", "match_number"])["start_date"]

def day(season, number):
    stamp = dated.get((season, number))
    return None if stamp is None else stamp.strftime("%Y-%m-%d")

pd.DataFrame([
    {"season": season, "missing match_number": n, "previous number played on": day(season, n - 1), "next number played on": day(season, n + 1)}
    for season, row in season_gaps.iterrows() for n in row["missing_numbers"]
])

,season,missing match_number,previous number played on,next number played on
0,2007/08,47,2008-05-21,2008-05-23
1,2009,7,2009-04-21,2009-04-22
2,2009,13,2009-04-25,2009-04-26
3,2011,20,2011-04-19,2011-04-20
4,2012,32,2012-04-24,2012-04-25
5,2012,34,2012-04-25,2012-04-26
6,2015,25,2015-04-25,2015-04-26
7,2017,29,2017-04-24,2017-04-26
8,2024,63,2024-05-12,2024-05-14
9,2024,66,2024-05-15,2024-05-17


**Reading D2**
- Gaps in **7 seasons** — 2007/08, 2009, 2011, 2012, 2015, 2017, 2024 — 10 missing numbers in all. That is all six seasons notebook 01 flagged by eye (58, 57, 73, 59, 59 and 71 matches) **plus 2012**, which looked round at 74 matches but whose numbering runs to 72 with 32 and 34 absent. The by-eye test missed 2012; the numbering test finds it.
- `match_number` increases with date inside a season, so each gap can be placed between its two neighbours (table above) — a starting point for looking the fixture up elsewhere.
- **What the method cannot see:**
  1. A missing match at the **end** of the league numbering — nothing later reveals it. 2024 may be an example: its numbering stops at 69, while 2011, 2022, 2023, 2025 and 2026 all run to 70. But that uses other seasons as a yardstick; the numbering itself shows no hole there.
  2. A missing **play-off** match: play-offs carry no number (D3), so they cannot leave a gap. Every season has 3 or 4 of them, which looks right but cannot prove completeness.
  3. **Why** a number is absent — abandoned without a ball, never collected, or lost. The files do not say.
- **Outside knowledge needed:** each season's official fixture list (league matches and play-offs) and which fixtures had no ball bowled. Cricsheet's own list of matches it does not cover (there is a "Missing" page in its site menu) is the first place to look; it was not consulted here.
- So a gap-free season is not necessarily complete: "no gaps" only means no hole *inside* the numbering.

### D3 · The 74 matches with no `match_number`

In [25]:
last_numbered = matches[matches["match_number"].notna()].groupby("season")["start_date"].max().rename("last numbered match")
unnumbered = (matches[matches["match_number"].isna()].groupby("season")["start_date"]
              .agg(["size", "min"]).rename(columns={"size": "unnumbered matches", "min": "first unnumbered match"}))
d3 = unnumbered.join(last_numbered)
d3["all start after the last numbered match"] = d3["first unnumbered match"] > d3["last numbered match"]
d3 = d3.join(season_gaps["year"]).sort_values("year").drop(columns="year")
print("unnumbered matches in total:", int(d3["unnumbered matches"].sum()))
d3

unnumbered matches in total: 74


,unnumbered matches,first unnumbered match,last numbered match,all start after the last numbered match
season,,,,
2007/08,3,2008-05-30,2008-05-28,True
2009,3,2009-05-22,2009-05-21,True
2009/10,4,2010-04-21,2010-04-19,True
2011,4,2011-05-24,2011-05-22,True
2012,4,2012-05-22,2012-05-20,True
2013,4,2013-05-21,2013-05-19,True
2014,4,2014-05-27,2014-05-25,True
2015,4,2015-05-19,2015-05-17,True
2016,4,2016-05-24,2016-05-22,True


**Reading D3.** The 74 matches without a `match_number` are the knockout (play-off) matches. What the data show: in every season they all start after the last numbered match, and each season has exactly 3 (2008, 2009) or 4 (2010 onwards) of them — 2 × 3 + 17 × 4 = 74. Calling them play-offs is the outside-knowledge part (the usual shape of an IPL season: a numbered league stage, then semi-finals and a final, later four play-off matches). The info files do not say *which* play-off each one is: `event` is constant and there is no stage key.

## What this notebook supersedes in notebook 01

| Notebook 01 | Now |
|---|---|
| §3.1 — six short seasons, "to test against the info files" | **Changed.** The info files cannot show absent matches directly (same 1,243 ids as the deliveries); numbering gaps can. 7 seasons have gaps — the six plus 2012 — and the method's blind spots are stated in D2. |
| §5 — low-count and single-innings matches, "cause not stated" | **Answered.** All six single-innings matches and the four lowest (22 / 51 / 63 / 63) are `no result`; 1178424 was a 5-over chase that never reached a result (D1). |
| Worklist 13 — super overs | **Confirmed.** `super_over` lines exist in exactly the 16 matches with innings > 2, and all 16 are `tie` + `eliminator` (validation table). |
| Worklist 14 — "flag short matches using result / method before excluding" | **Now possible.** Result types: plain win 1,195 · D/L win 23 · tie decided by super over 16 · no result 9 (C2). Who won = `winner`, else `eliminator`. |
| Worklist 18 — `actual_delivery`, `non_boundary`, `fielder_1–3` | **Answered.** [Cricsheet's format page](https://cricsheet.org/format/csv_ashwin/) defines all five (added in format 2.3.0, which is why the older bundled README lists only 22 columns). It documents `ball` and `actual_delivery` as *strings*, which supports the float-`ball` finding; `non_boundary` is a 4 or 6 that was not scored via an actual boundary (all run, overthrows); `actual_delivery` repeats across a wide or no-ball, as found. |
| §3.3 / worklist 5–6 — "city cannot be read from `venue` alone" | **Changed, with a correction.** The info files carry a `city` for 1,192 of 1,243 matches (37 distinct values); the other 51 are blank — all at the Dubai and Sharjah grounds. The 37 values need their own check in the cleaning session (`Bangalore` and `Bengaluru` both occur). An earlier draft of this table said every match has a city; that was wrong. |
| Worklist 1–6 — standardising season, team and venue strings | **Unchanged.** The info files use the same strings: season, venue and the team pair agree with the deliveries match by match (validation table), so nothing here fixes them. |
| New in this notebook | `team1` / `team2` order is not a stable convention (C2). Target overs are written `over.ball` (C3). D/L results do not reconcile with raw totals but do with `target_runs` (C4). |